In [ ]:
from transformers.models.auto.modeling_auto import AutoModelForCausalLM
from transformers.models.auto.tokenization_auto import AutoTokenizer
import torch
from cs336_alignment.my_sft_utils import log_generations

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    "/home/nova/cs336/assignment5-alignment/models/Qwen2.5-Math-1.5B",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
tokenizer = AutoTokenizer.from_pretrained("/home/nova/cs336/assignment5-alignment/models/Qwen2.5-Math-1.5B")
tokenizer.padding_side = "left"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotary_emb): Qw

In [5]:

prompts = [
    "Explain what is supervised fine-tuning.",
    "Compute 2 + 3.",
]
ground_truths = [
    "Supervised fine-tuning adjusts model weights using labeled pairs.",
    "5",
]


In [6]:

def reward_fn(resp: str, gt: str):
    is_correct = resp.strip() == gt.strip()
    return {"format": "simple", "answer": resp, "total_reward": 1.0 if is_correct else 0.0, "is_correct": is_correct}



In [9]:


result = log_generations(
    model,
    tokenizer,
    prompts,
    ground_truths,
    reward_fn,
    max_new_tokens=32,
)
print(result)


{'samples': [{'prompt': 'Explain what is supervised fine-tuning.', 'response': ' What is the difference between supervised fine-tuning and fine-tuning? Supervised fine-tuning is a technique used in machine learning, particularly in the context of', 'ground_truth': 'Supervised fine-tuning adjusts model weights using labeled pairs.', 'reward': {'format': 'simple', 'answer': ' What is the difference between supervised fine-tuning and fine-tuning? Supervised fine-tuning is a technique used in machine learning, particularly in the context of', 'total_reward': 0.0, 'is_correct': False}, 'token_entropy': 1.053002119064331, 'response_length': 32, 'is_correct': False}, {'prompt': 'Compute 2 + 3.', 'response': '4 - 0.4^2. To solve the expression \\(2 + 3.4 - 0.4^2\\), we can break', 'ground_truth': '5', 'reward': {'format': 'simple', 'answer': '4 - 0.4^2. To solve the expression \\(2 + 3.4 - 0.4^2\\), we can break', 'total_reward': 0.0, 'is_correct': False}, 'token_entropy': 0.6305886507034302, 

In [ ]:
input_ids = train_batch["input_ids"].to(device)
labels = train_batch["labels"].to(device)
logits = model(input_ids).logits
loss = F.cross_entropy(..., ...)

In [ ]:
model.save_pretrained(save_directory=output_dir)
tokenizer.save_pretrained(save_directory=output_dir)

### SFT microbatch train step

We are now ready to implement a single microbatch train step for SFT (recall that for a train minibatch, we iterate over many microbatches if `gradient_accumulation_steps > 1`).

### Problem (sft_microbatch_train_step): Microbatch train step (3 points)

**Deliverable:** Implement a single micro-batch update for SFT, including cross-entropy loss, summing with a mask, and gradient scaling.

**Implementation tips:**
- You should call `loss.backward()` in this function. Make sure to adjust for gradient accumulation.

To test your code, implement `adapters.run_sft_microbatch_train_step`. Then run `uv run pytest -k test_sft_microbatch_train_step` and confirm it passes.


In [5]:
import torch
tensor = torch.rand(2,3,4)
tensor

tensor([[[0.0595, 0.0113, 0.7153, 0.4616],
         [0.8453, 0.1704, 0.5957, 0.5279],
         [0.8438, 0.8922, 0.7438, 0.0517]],

        [[0.4902, 0.6290, 0.6156, 0.1650],
         [0.8730, 0.0020, 0.7548, 0.0221],
         [0.4982, 0.4980, 0.7558, 0.6050]]])

In [10]:
red = torch.sum(tensor,dim=-1)

In [11]:
red.mean()

tensor(1.9712)

In [12]:
import torch
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

12.4
NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
from cs336_alignment.vllm_work_with_hf import init_vllm, load_policy_into_vllm_instance
model = AutoModelForCausalLM.from_pretrained(
    "/home/nova/cs336/assignment5-alignment/models/Qwen2.5-Math-1.5B",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
tokenizer = AutoTokenizer.from_pretrained("/home/nova/cs336/assignment5-alignment/models/Qwen2.5-Math-1.5B")

llm = init_vllm(
    model_id="/home/nova/cs336/assignment5-alignment/models/Qwen2.5-Math-1.5B",
    device="cuda:1",
    seed=42,
    gpu_memory_utilization=0.85,
)
load_policy_into_vllm_instance(model, llm)